In [3]:
import cv2 as cv
import mediapipe as mp
import numpy as np
import joblib

# Loading the model
model = joblib.load('Random_forest_focus_model.pkl')

CLASSES = {
    0 : "Focused",
    1 : "Head Turned",
    2 : "Looking Down",
    3 : "Eyes Distracted",
    4 : "Drowsy",
}

# Loading the facemesh

mp_face_mesh = mp.solutions.face_mesh
face_mesh = mp_face_mesh.FaceMesh(refine_landmarks=True)

cap = cv.VideoCapture(0)

while cap.isOpened():
    success, frame = cap.read()
    if not success:
        break
    
    frame = cv.flip(frame,1)
    rgb_frame = cv.cvtColor(frame,cv.COLOR_BGR2RGB)
    
    results = face_mesh.process(rgb_frame)
    
    if results.multi_face_landmarks:
        for face_landmarks in results.multi_face_landmarks:
            landmarks = []
            
            for lm in face_landmarks.landmark:
                landmarks.extend([lm.x,lm.y,lm.z])
                
            landmarks = np.array(landmarks).reshape(-1,3)
            nose = landmarks[4]
            
            landmarks = landmarks - nose
            dist = np.linalg.norm(landmarks[33] - landmarks[263])
            landmarks = landmarks/dist
            normalized_input = landmarks.flatten().reshape(1, -1)
            
            # Predicting 
            
            prediction = model.predict(normalized_input)[0]
            probability = np.max(model.predict_proba(normalized_input))
            
            status = CLASSES[prediction]
            
            # Visuals
            color = (0,255,0) if prediction == 0 else (0,0,255)
            cv.putText(frame,f"Status: {status} ({probability:0.2f})",(20,50),cv.FONT_HERSHEY_COMPLEX,1,color,3)
            
    cv.imshow('Real-Time Focus Tracker', frame)
    key = cv.waitKey(1) & 0xFF
    if key == ord('q'): 
        break
                
cap.release()
cv.destroyAllWindows()

c:\Users\User\Desktop\OpenCV\venv\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
c:\Users\User\Desktop\OpenCV\venv\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
c:\Users\User\Desktop\OpenCV\venv\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
c:\Users\User\Desktop\